In [1]:
import numpy as np
import pandas as pd

In [2]:
embeddings_dir = "/runs/20260113_2024v2-embeddings_2024v2-model_all"

embeddings = np.load(f"{embeddings_dir}/embeddings_raw.npy")
img_paths = pd.read_csv(f"{embeddings_dir}/image_paths.csv", header=None)[0].tolist()
labels = np.load(f"{embeddings_dir}/labels.npy")
label_names = np.load(f"{embeddings_dir}/label_names.npy", allow_pickle=True)
metadata = pd.read_csv("/experiments/dhruv_4D_ResnetBiLSTM_phate/2024v2_metadata/2024v2_phate_metadata.csv")

In [12]:
# metadata = metadata[metadata['path'].str.contains("-0.npy")]
metadata['label_name'] = ''
metadata['label_id'] = -1
metadata['embedding'] = None
metadata['time'] = -1.

In [13]:
metadata.head()

,sample id,region id,cell id,movie id,global_start_frame,path,label_name,label_id,embedding,time
0,20240729-1,0,0,0,0,20240729-1/000000-0.npy,,-1,None,-1.0
1,20240729-1,0,1,0,0,20240729-1/000001-0.npy,,-1,None,-1.0
2,20240729-1,0,2,0,0,20240729-1/000002-0.npy,,-1,None,-1.0
3,20240729-1,0,3,0,0,20240729-1/000003-0.npy,,-1,None,-1.0
4,20240729-1,0,4,0,0,20240729-1/000004-0.npy,,-1,None,-1.0


array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

In [6]:
for i, row in metadata.iterrows():
    base_path = row['path']
    matching_indices = [j for j, path in enumerate(img_paths) if base_path in path]
    if matching_indices:
        idx = matching_indices[0]
        metadata.at[i, 'label_name'] = label_names[labels[idx]]
        metadata.at[i, 'label_id'] = labels[idx]
        metadata.at[i, 'embedding'] = embeddings[idx]

In [11]:
len(metadata)

34272

In [7]:
# Average per movie id per region per condition and stack regions for per-condition trajectories
phate_data = []

for lbl in np.unique(labels):
    df_condition = metadata[metadata['label_id'] == lbl].reset_index(drop=True)
    label_name = df_condition.at[0, 'label_name']
    regions = df_condition['region id'].unique()
    for region in regions:
        df_region = df_condition[df_condition['region id'] == region].reset_index(drop=True)

        movie_ids = df_region['movie id'].unique()
        for movie_id in movie_ids:
            df_movie = df_region[df_region['movie id'] == movie_id].reset_index(drop=True)
            embeddings = np.mean(np.stack(df_movie['embedding'].to_list()), axis=0)

            t_start = df_movie.at[0, 'global_start_frame']
            for i in range(len(embeddings)):
                t = t_start + i
                phate_data.append({"drug": label_name, "time": float(t), "embedding": embeddings[i]})

df_phate = pd.DataFrame(phate_data)

In [8]:
df_movie.head()

,sample id,region id,cell id,movie id,global_start_frame,path,label_name,label_id,embedding
0,20240913-1,9,1581,0,540,20240913-1/001581-0.npy,paraquat,25,"[[-0.00024672793, 0.008640074, -0.009963654, 0..."
1,20240913-1,9,1592,0,540,20240913-1/001592-0.npy,paraquat,25,"[[-0.0061015217, 0.0047693076, -0.015569459, 0..."
2,20240913-1,9,1620,0,540,20240913-1/001620-0.npy,paraquat,25,"[[0.004153649, 0.0023767399, 0.0019207133, 0.0..."
3,20240913-1,9,1635,0,540,20240913-1/001635-0.npy,paraquat,25,"[[0.0007418606, 0.0032168988, -0.0058588115, 0..."
4,20240913-1,9,1642,0,540,20240913-1/001642-0.npy,paraquat,25,"[[-0.0013874967, -0.0020052989, 0.000810216, 0..."


In [9]:
# Save to parquet
df_phate.to_parquet("2024v2-val_4DMS-2024v2_phate_mean-pooled.parquet", engine='pyarrow', index=False)